In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit

In [2]:
data_log2 = pd.read_pickle('data_log2_Lisbon_Coimbra.pkl')

In [3]:
data_log2

Protein.Group,A0A0A0MS15,A0A0B4J1X8,A0A0C4DH38,O00391,O00533,O14498,O14594,O15394,O43505,O75326,...,Q99574,Q9NQ79,Q9NRN5,Q9P121,Q9P2S2,Q9UBP4,Q9ULB1,Q9Y4C0,Q9Y646,Q9Y6R7
LIS090,7.524839,9.440618,7.131415,6.264067,8.355800,6.222868,5.798631,7.860578,9.961581,7.550508,...,6.205688,8.871862,4.881704,8.309122,6.827273,10.918669,7.058630,7.842011,5.686817,8.661849
LIS098,7.791541,10.339338,7.438942,5.908674,7.962676,5.939913,5.244541,7.345982,10.132975,7.260788,...,6.045196,8.902517,4.443076,8.087309,6.454133,10.520913,6.675844,7.766999,5.921993,7.172317
LIS017,6.309827,10.338246,6.398735,5.907285,8.112664,5.983963,6.077408,7.673055,10.326800,7.580711,...,6.098369,8.842668,5.340762,8.334036,6.557727,10.662704,6.746998,7.561639,6.478499,6.753203
LIS007,9.705954,11.126569,8.909980,6.499706,7.305670,6.535481,4.455360,7.376681,9.405173,7.268491,...,5.996023,8.386212,4.832850,7.650348,6.108621,10.128703,6.173705,6.919900,6.039838,8.260840
LIS026,8.114325,11.020813,8.179367,6.765097,8.106155,5.791983,5.600198,7.862365,9.843741,7.020424,...,6.037775,8.941429,5.365864,8.227313,6.172213,10.573581,6.740347,7.526218,5.885477,7.837615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109089,8.077537,10.276252,6.114536,6.263756,8.592278,6.632142,5.617401,7.627045,10.475673,6.919173,...,5.550356,8.981245,4.746759,8.031175,6.308907,11.535426,6.838952,7.384663,5.610963,8.048367
103176,7.166143,9.615435,6.836252,5.507424,8.735336,6.767973,6.649041,7.817770,10.861289,7.066046,...,5.305066,8.930687,4.078755,8.127499,6.688908,11.910107,6.889108,7.403089,6.744659,7.426063
102357,7.540244,9.095864,6.917348,6.168650,8.682271,6.479260,6.619081,7.842086,10.577948,7.061927,...,5.163386,9.077726,3.872740,8.187308,6.619353,11.328832,6.974518,7.468730,5.825379,7.680205
107362,6.883413,8.918684,6.115443,5.905714,8.505625,5.880857,6.616853,7.870044,10.711813,6.709373,...,5.301803,8.996601,4.641332,8.406277,6.488221,11.888595,6.811664,7.398692,5.688082,7.774392


In [4]:
import pickle

with open('list_groups_LC_2.pkl', 'rb') as f:
    list_groups = pickle.load(f)

print(list_groups)

['MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT',

In [5]:
list_groups = pd.Series(list_groups)
print(len(list_groups))
data_log2 = data_log2.iloc[:,:-4]

128


In [6]:
df_all_data = pd.read_pickle('df_all_data_Lisbon_Coimbra.pkl')
df_Coimbra = df_all_data[df_all_data['Lab_name'] == 'Coimbra']
y_Coimbra = df_Coimbra['Groups']
df_Lisbon = df_all_data[df_all_data['Lab_name'] == 'Lisbon']
y_Lisbon = df_Lisbon['Groups']
cols_to_drop = ['Lab', 'Sample code', 'Groups', 'Age', 'Gender', 'Lab_name']
df_coimbra_dc = df_Coimbra.drop(columns=cols_to_drop)
df_Lisbon_dc = df_Lisbon.drop(columns=cols_to_drop)
print("Same features:", all(df_coimbra_dc.columns == df_Lisbon_dc.columns))

Same features: True


In [7]:
'''
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold

# ======================
# DATA SETUP
# ======================
X = data_log2.copy()
y = np.array(list_groups)
y_binary = (y == "MCI-AD").astype(int)

# ======================
# EXPERIMENT PARAMETERS
# ======================
k_values = [5, 10, 15, 20]
seeds = range(5)
all_k_results = []

print("Starting Sensitivity Analysis for K-features...")

# ======================
# OUTER LOOP: K-VALUES
# ======================
for k in k_values:
    k_seeds_mcc = []
    k_seeds_auc = []
    k_feature_union = set()

    print(f"\nTesting K = {k} proteins...")

    for seed in seeds:
        # 1. Split 70/30
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, stratify=y, random_state=seed
        )
        y_test_bin = (y_test == "MCI-AD").astype(int)

        # 2. Simple Feature Importance (5-Fold CV on Train only)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        fold_importances = []

        for train_idx, val_idx in cv.split(X_train, y_train):
            X_sub, y_sub = X_train.iloc[train_idx], y_train[train_idx]
            rf_fs = RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=seed)
            rf_fs.fit(X_sub, y_sub)
            fold_importances.append(rf_fs.feature_importances_)

        # Average importance and Ranking
        mean_imp = np.mean(fold_importances, axis=0)
        ranking = pd.DataFrame({"protein": X.columns, "imp": mean_imp}).sort_values("imp", ascending=False)
        
        # Select Top-K
        top_features = ranking["protein"].iloc[:k].tolist()
        k_feature_union.update(top_features)

        # 3. Final Model Evaluation
        rf_final = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42)
        rf_final.fit(X_train[top_features], y_train)

        # Predictions
        y_pred = rf_final.predict(X_test[top_features])
        ad_idx = np.where(rf_final.classes_ == "MCI-AD")[0][0]
        y_prob = rf_final.predict_proba(X_test[top_features])[:, ad_idx]

        k_seeds_mcc.append(matthews_corrcoef(y_test, y_pred))
        k_seeds_auc.append(roc_auc_score(y_test_bin, y_prob))

    # Save aggregated results for this K
    all_result = {
        "K": k,
        "mean_mcc": np.mean(k_seeds_mcc),
        "std_mcc": np.std(k_seeds_mcc),
        "mean_auc": np.mean(k_seeds_auc),
        "std_auc": np.std(k_seeds_auc),
        "union_size": len(k_feature_union)
    }
    all_results.append(res)
    
    # INDIVIDUAL REPORT FOR EACH K
    print(f"--- REPORT FOR K={k} ---")
    print(f"Mean MCC: {res['mean_mcc']:.3f} (±{res['std_mcc']:.3f})")
    print(f"Mean AUC: {res['mean_auc']:.3f} (±{res['std_auc']:.3f})")
    print(f"Total Unique Proteins in Union: {res['union_size']}")

# ======================
# FINAL COMPARISON TABLE
# ======================
df_final = pd.DataFrame(all_results)
print("\n" + "="*50)
print("FINAL SENSITIVITY ANALYSIS SUMMARY")
print("="*50)
print(df_final[["K", "mean_mcc", "mean_auc", "union_size"]].to_string(index=False))
'''

'\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import matthews_corrcoef, roc_auc_score\nfrom sklearn.model_selection import train_test_split, StratifiedKFold\n\n# ======================\n# DATA SETUP\n# ======================\nX = data_log2.copy()\ny = np.array(list_groups)\ny_binary = (y == "MCI-AD").astype(int)\n\n# ======================\n# EXPERIMENT PARAMETERS\n# ======================\nk_values = [5, 10, 15, 20]\nseeds = range(5)\nall_k_results = []\n\nprint("Starting Sensitivity Analysis for K-features...")\n\n# ======================\n# OUTER LOOP: K-VALUES\n# ======================\nfor k in k_values:\n    k_seeds_mcc = []\n    k_seeds_auc = []\n    k_feature_union = set()\n\n    print(f"\nTesting K = {k} proteins...")\n\n    for seed in seeds:\n        # 1. Split 70/30\n        X_train, X_test, y_train, y_test = train_test_split(\n            X, y, test_size=0.3, stratify=y, random_state=seed\n        )\n 

Training: Coimbra Dataset | Testing: Lisbon Dataset
This section describes the application of Partial Least Squares Discriminant Analysis (PLS-DA) to evaluate the generalizability of the protein signature across different clinical centers (multicentric validation).

In [8]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit

# =================================================================
# DATA PREPARATION (Coimbra as Training | Lisbon as External Test)
# =================================================================

print('ok')
X_train = df_coimbra_dc.copy()
y_train = np.array(y_Coimbra)

X_test = df_Lisbon_dc.copy()
y_test = np.array(y_Lisbon)

y_train_bin = (y_train == "MCI-AD").astype(int)
y_test_bin = (y_test == "MCI-AD").astype(int)

# ======================
# GLOBAL PARAMETERS
# ======================
thresholds = [0.7, 0.8, 0.9]
n_iterations = 20
seeds = range(15)

# STORAGE
all_results = []
feature_counter = {}   # To track frequency of EVERY selected protein
threshold_counter = {} # To track frequency of best thresholds

def stratified_bootstrap(X, y):
    ad_idx = np.where(y == "MCI-AD")[0]
    ct_idx = np.where(y == "MCI-CT")[0]
    ad_sample = np.random.choice(ad_idx, size=len(ad_idx), replace=True)
    ct_sample = np.random.choice(ct_idx, size=len(ct_idx), replace=True)
    indices = np.concatenate([ad_sample, ct_sample])
    np.random.shuffle(indices)
    return X.iloc[indices], y[indices]

# ======================
# SEED LOOP
# ======================
for seed in seeds:
    # Internal CV setup
    cv = StratifiedShuffleSplit(n_splits=5, test_size=0.3, random_state=seed)
    threshold_scores = []

    for t in thresholds:
        fold_scores = []
        for train_idx, val_idx in cv.split(X_train, y_train):
            X_subtrain, y_subtrain = X_train.iloc[train_idx], y_train[train_idx]
            X_val, y_val = X_train.iloc[val_idx], y_train[val_idx]

            # Bootstrap Feature Importance
            feature_names = X_subtrain.columns
            importance_matrix = np.zeros((n_iterations, len(feature_names)))

            for i in range(n_iterations):
                X_boot, y_boot = stratified_bootstrap(X_subtrain, y_subtrain)
                rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=seed)
                rf.fit(X_boot, y_boot)
                importance_matrix[i] = rf.feature_importances_

            mean_importance = importance_matrix.mean(axis=0)
            ranking_df = pd.DataFrame({"protein": feature_names, "importance": mean_importance}).sort_values("importance", ascending=False)
            ranking_df["importance"] /= ranking_df["importance"].sum()
            ranking_df["cum_importance"] = ranking_df["importance"].cumsum()

            selected_features = ranking_df[ranking_df["cum_importance"] <= t]["protein"]
            if len(selected_features) == 0: selected_features = ranking_df["protein"].iloc[:1]

            rf_fold = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=seed)
            rf_fold.fit(X_subtrain[selected_features], y_subtrain)
            y_pred = rf_fold.predict(X_val[selected_features])
            fold_scores.append(matthews_corrcoef(y_val, y_pred))

        threshold_scores.append(np.mean(fold_scores))

    # Best threshold for this seed
    best_threshold = thresholds[np.argmax(threshold_scores)]
    threshold_counter[best_threshold] = threshold_counter.get(best_threshold, 0) + 1
    
    # Final Training for this seed
    importance_matrix_final = np.zeros((n_iterations, len(X_train.columns)))
    for i in range(n_iterations):
        X_boot, y_boot = stratified_bootstrap(X_train, y_train)
        rf = RandomForestClassifier(n_estimators=500, class_weight="balanced", n_jobs=-1, random_state=i)
        rf.fit(X_boot, y_boot)
        importance_matrix_final[i] = rf.feature_importances_

    mean_imp_final = importance_matrix_final.mean(axis=0)
    ranking_final = pd.DataFrame({"protein": X_train.columns, "importance": mean_imp_final}).sort_values("importance", ascending=False)
    ranking_final["importance"] /= ranking_final["importance"].sum()
    ranking_final["cum_importance"] = ranking_final["importance"].cumsum()

    top_features = ranking_final[ranking_final["cum_importance"] <= best_threshold]["protein"]
    
    # TRACK EVERY FEATURE SELECTED IN THIS SEED
    for f in top_features:
        feature_counter[f] = feature_counter.get(f, 0) + 1

    # External Test (Lisbon)
    rf_final = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42)
    rf_final.fit(X_train[top_features], y_train)
    y_pred_lisbon = rf_final.predict(X_test[top_features])
    ad_idx_label = np.where(rf_final.classes_ == "MCI-AD")[0][0]
    y_prob_lisbon = rf_final.predict_proba(X_test[top_features])[:, ad_idx_label]

    seed_mcc = matthews_corrcoef(y_test, y_pred_lisbon)
    seed_auc = roc_auc_score(y_test_bin, y_prob_lisbon)

    # PER-SEED REPORT
    print(f"\n>> SEED {seed} RESULTS:")
    print(f"   Best Threshold: {best_threshold}")
    print(f"   Features Selected: {len(top_features)}")
    print(f"   Test MCC: {seed_mcc:.3f}")
    print(f"   Test AUC: {seed_auc:.3f}")

    all_results.append({
        "model": "RF","seed": seed, "mcc": seed_mcc, "auc": seed_auc, 
        "n_features": len(top_features), "threshold": best_threshold
    })

# ======================
# FINAL AGGREGATED REPORT
# ======================
df_res = pd.DataFrame(all_results)

print("\n" + "="*30)
print("FINAL SUMMARY (OVER ALL SEEDS)")
print("="*30)
print(f"Global Mean MCC: {df_res['mcc'].mean():.3f} (±{df_res['mcc'].std():.3f})")
print(f"Global Mean AUC: {df_res['auc'].mean():.3f} (±{df_res['auc'].std():.3f})")
print(f"Avg Features Selected: {df_res['n_features'].mean():.1f}")

print("\n--- Feature Frequency (Top 20) ---")
# Normalized frequency (0.0 to 1.0)
feat_freq_CoimbraonLisbon = (pd.Series(feature_counter).sort_values(ascending=False) / len(seeds))
print(feat_freq_CoimbraonLisbon.head(20))

print("\n--- Threshold Frequency ---")
thresh_freq = (pd.Series(threshold_counter).sort_index() / len(seeds))
print(thresh_freq)

ok

>> SEED 0 RESULTS:
   Best Threshold: 0.9
   Features Selected: 82
   Test MCC: 0.305
   Test AUC: 0.736

>> SEED 1 RESULTS:
   Best Threshold: 0.7
   Features Selected: 34
   Test MCC: 0.323
   Test AUC: 0.703

>> SEED 2 RESULTS:
   Best Threshold: 0.7
   Features Selected: 33
   Test MCC: 0.323
   Test AUC: 0.683

>> SEED 3 RESULTS:
   Best Threshold: 0.9
   Features Selected: 80
   Test MCC: 0.323
   Test AUC: 0.744

>> SEED 4 RESULTS:
   Best Threshold: 0.9
   Features Selected: 80
   Test MCC: 0.305
   Test AUC: 0.726

>> SEED 5 RESULTS:
   Best Threshold: 0.9
   Features Selected: 83
   Test MCC: 0.323
   Test AUC: 0.728

>> SEED 6 RESULTS:
   Best Threshold: 0.8
   Features Selected: 51
   Test MCC: 0.323
   Test AUC: 0.709

>> SEED 7 RESULTS:
   Best Threshold: 0.7
   Features Selected: 33
   Test MCC: 0.341
   Test AUC: 0.718

>> SEED 8 RESULTS:
   Best Threshold: 0.7
   Features Selected: 34
   Test MCC: 0.288
   Test AUC: 0.682

>> SEED 9 RESULTS:
   Best Threshold: 0.7


Dataset: Coimbra + Lisbon (Merged)
In this section, a Partial Least Squares Discriminant Analysis (Random Forest) is performed on the combined clinical proteomic dataset.

In [9]:
print(len(feat_freq_CoimbraonLisbon))
feat_freq_CoimbraonLisbon.to_pickle('feat_freq_CoimbraonLisbon_rf_threshold.pkl')

97


In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import matthews_corrcoef, roc_auc_score

# ======================
# DATI
# ======================
X = data_log2.copy()
y = np.array(list_groups)

# binarizza per AUC (importante!)
y_binary = (y == "MCI-AD").astype(int)

# ======================
# PARAMETRI GLOBALI
# ======================
thresholds = [ 0.7, 0.8, 0.9]
n_iterations = 20
seeds = range(15)

# ======================
# STORAGE
# ======================
all_results = []
feature_counter = {}
threshold_counter = {}

# ======================
# BOOTSTRAP
# ======================
def stratified_bootstrap(X, y):
    ad_idx = np.where(y == "MCI-AD")[0]
    ct_idx = np.where(y == "MCI-CT")[0]

    ad_sample = np.random.choice(ad_idx, size=len(ad_idx), replace=True)
    ct_sample = np.random.choice(ct_idx, size=len(ct_idx), replace=True)

    indices = np.concatenate([ad_sample, ct_sample])
    np.random.shuffle(indices)

    return X.iloc[indices], y[indices]


# ======================
# LOOP SEED
# ======================
for seed in seeds:

    print(f"\n===== SEED {seed} =====")

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=seed
    )

    y_train_bin = (y_train == "MCI-AD").astype(int)
    y_test_bin = (y_test == "MCI-AD").astype(int)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    threshold_scores = []

    # ======================
    # CV per scegliere threshold
    # ======================
    for t in thresholds:

        fold_scores = []

        for train_idx, val_idx in cv.split(X_train, y_train):

            X_subtrain = X_train.iloc[train_idx]
            y_subtrain = y_train[train_idx]

            X_val = X_train.iloc[val_idx]
            y_val = y_train[val_idx]

            y_val_bin = (y_val == "MCI-AD").astype(int)

            # ===== bootstrap importance
            feature_names = X_subtrain.columns
            importance_matrix = np.zeros((n_iterations, len(feature_names)))

            for i in range(n_iterations):

                X_boot, y_boot = stratified_bootstrap(X_subtrain, y_subtrain)

                rf = RandomForestClassifier(
                    n_estimators=200,
                    class_weight="balanced",
                    n_jobs=-1,
                    random_state=seed
                )

                rf.fit(X_boot, y_boot)
                importance_matrix[i] = rf.feature_importances_

            mean_importance = importance_matrix.mean(axis=0)

            ranking_df = pd.DataFrame({
                "protein": feature_names,
                "importance": mean_importance
            }).sort_values("importance", ascending=False)

            # normalizzazione
            ranking_df["importance"] /= ranking_df["importance"].sum()
            ranking_df["cum_importance"] = ranking_df["importance"].cumsum()

            # selezione feature
            selected_features = ranking_df[
                ranking_df["cum_importance"] <= t
            ]["protein"]

            if len(selected_features) == 0:
                selected_features = ranking_df["protein"].iloc[:1]

            # train modello
            rf = RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                random_state=seed
            )

            rf.fit(X_subtrain[selected_features], y_subtrain)

            # valutazione
            y_pred = rf.predict(X_val[selected_features])
            mcc = matthews_corrcoef(y_val, y_pred)

            fold_scores.append(mcc)

        threshold_scores.append(np.mean(fold_scores))

    # ======================
    # best threshold
    # ======================
    best_threshold = thresholds[np.argmax(threshold_scores)]
    print("Best threshold:", best_threshold)

    # ======================
    # TRAIN FINALE (su tutto train)
    # ======================
    feature_names = X_train.columns
    importance_matrix = np.zeros((n_iterations, len(feature_names)))

    for i in range(n_iterations):

        X_boot, y_boot = stratified_bootstrap(X_train, y_train)

        rf = RandomForestClassifier(
            n_estimators=500,
            class_weight="balanced",
            n_jobs=-1,
            random_state=i
        )

        rf.fit(X_boot, y_boot)
        importance_matrix[i] = rf.feature_importances_

    mean_importance = importance_matrix.mean(axis=0)

    ranking_df = pd.DataFrame({
        "protein": feature_names,
        "importance": mean_importance
    }).sort_values("importance", ascending=False)

    ranking_df["importance"] /= ranking_df["importance"].sum()
    ranking_df["cum_importance"] = ranking_df["importance"].cumsum()

    top_features = ranking_df[
        ranking_df["cum_importance"] <= best_threshold
    ]["protein"]

    if len(top_features) == 0:
        top_features = ranking_df["protein"].iloc[:1]

    print("Numero feature:", len(top_features))

    # ======================
    # TEST
    # ======================
    rf_final = RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42
    )

    rf_final.fit(X_train[top_features], y_train)

    y_pred = rf_final.predict(X_test[top_features])
    class_order = rf_final.classes_
    ad_index = np.where(class_order == "MCI-AD")[0][0]
    
    y_prob = rf_final.predict_proba(X_test[top_features])[:, ad_index]

    test_mcc = matthews_corrcoef(y_test, y_pred)
    test_auc = roc_auc_score(y_test_bin, y_prob)
    print("Class order:", rf_final.classes_)
    print("Test MCC:", test_mcc)
    print("Test AUC:", test_auc)

    # ======================
    # SALVATAGGIO
    # ======================
    all_results.append({
        "seed": seed,
        "mcc": test_mcc,
        "auc": test_auc,
        "n_features": len(top_features),
        "best_threshold": best_threshold
    })

    # frequenza feature
    for f in top_features:
        feature_counter[f] = feature_counter.get(f, 0) + 1

    # frequenza threshold
    threshold_counter[best_threshold] = threshold_counter.get(best_threshold, 0) + 1


# ======================
# RISULTATI FINALI
# ======================
df_results = pd.DataFrame(all_results)

print("\n===== FINAL RESULTS =====")
print("Mean MCC:", df_results["mcc"].mean())
print("Mean AUC:", df_results["auc"].mean())

# frequenze
feature_freq = pd.Series(feature_counter).sort_values(ascending=False) / len(seeds)
threshold_freq = pd.Series(threshold_counter).sort_index() / len(seeds)

print("\nTop features (frequency):")
print(feature_freq.head(20))

print("\nThreshold frequency:")
print(threshold_freq)


===== SEED 0 =====
Best threshold: 0.9
Numero feature: 113
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.8539125638299666
Test AUC: 1.0

===== SEED 1 =====
Best threshold: 0.8
Numero feature: 74
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.9258200997725515
Test AUC: 0.9583333333333334

===== SEED 2 =====
Best threshold: 0.9
Numero feature: 109
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.8539125638299666
Test AUC: 0.9702380952380952

===== SEED 3 =====
Best threshold: 0.7
Numero feature: 45
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.8571428571428571
Test AUC: 0.943452380952381

===== SEED 4 =====
Best threshold: 0.8
Numero feature: 73
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.9249624617007739
Test AUC: 1.0

===== SEED 5 =====
Best threshold: 0.9
Numero feature: 117
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.6904761904761905
Test AUC: 0.9702380952380952

===== SEED 6 =====
Best threshold: 0.9
Numero feature: 113
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.8539125638299666
Test A

In [11]:
print(len(feature_freq))
feature_freq.to_pickle('feature_freq_LC_rf.pkl')

143


Training: Lisbon Dataset | Testing: Coimbra Dataset
This section describes the "Inverse Validation" approach using Partial Least Squares Discriminant Analysis (PLS-DA). By swapping the roles of the two cohorts, we test the consistency and symmetry of our protein biomarkers.

In [12]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit

# =================================================================
# DATA PREPARATION (Lisbon as Training | Coimbra as External Test)
# =================================================================
print('ok')
X_train = df_Lisbon_dc.copy()
y_train = np.array(y_Lisbon)

X_test = df_coimbra_dc.copy()
y_test = np.array(y_Coimbra)

y_train_bin = (y_train == "MCI-AD").astype(int)
y_test_bin = (y_test == "MCI-AD").astype(int)

# ======================
# GLOBAL PARAMETERS
# ======================
thresholds = [0.7, 0.8, 0.9]
n_iterations = 20
seeds = range(15)

# STORAGE
all_results = []
feature_counter = {}   # To track frequency of EVERY selected protein
threshold_counter = {} # To track frequency of best thresholds

def stratified_bootstrap(X, y):
    ad_idx = np.where(y == "MCI-AD")[0]
    ct_idx = np.where(y == "MCI-CT")[0]
    ad_sample = np.random.choice(ad_idx, size=len(ad_idx), replace=True)
    ct_sample = np.random.choice(ct_idx, size=len(ct_idx), replace=True)
    indices = np.concatenate([ad_sample, ct_sample])
    np.random.shuffle(indices)
    return X.iloc[indices], y[indices]

# ======================
# SEED LOOP
# ======================
for seed in seeds:
    # Internal CV setup
    cv = StratifiedShuffleSplit(n_splits=5, test_size=0.3, random_state=seed)
    threshold_scores = []

    for t in thresholds:
        fold_scores = []
        for train_idx, val_idx in cv.split(X_train, y_train):
            X_subtrain, y_subtrain = X_train.iloc[train_idx], y_train[train_idx]
            X_val, y_val = X_train.iloc[val_idx], y_train[val_idx]

            # Bootstrap Feature Importance
            feature_names = X_subtrain.columns
            importance_matrix = np.zeros((n_iterations, len(feature_names)))

            for i in range(n_iterations):
                X_boot, y_boot = stratified_bootstrap(X_subtrain, y_subtrain)
                rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=seed)
                rf.fit(X_boot, y_boot)
                importance_matrix[i] = rf.feature_importances_

            mean_importance = importance_matrix.mean(axis=0)
            ranking_df = pd.DataFrame({"protein": feature_names, "importance": mean_importance}).sort_values("importance", ascending=False)
            ranking_df["importance"] /= ranking_df["importance"].sum()
            ranking_df["cum_importance"] = ranking_df["importance"].cumsum()

            selected_features = ranking_df[ranking_df["cum_importance"] <= t]["protein"]
            if len(selected_features) == 0: selected_features = ranking_df["protein"].iloc[:1]

            rf_fold = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=seed)
            rf_fold.fit(X_subtrain[selected_features], y_subtrain)
            y_pred = rf_fold.predict(X_val[selected_features])
            fold_scores.append(matthews_corrcoef(y_val, y_pred))

        threshold_scores.append(np.mean(fold_scores))

    # Best threshold for this seed
    best_threshold = thresholds[np.argmax(threshold_scores)]
    threshold_counter[best_threshold] = threshold_counter.get(best_threshold, 0) + 1
    
    # Final Training for this seed
    importance_matrix_final = np.zeros((n_iterations, len(X_train.columns)))
    for i in range(n_iterations):
        X_boot, y_boot = stratified_bootstrap(X_train, y_train)
        rf = RandomForestClassifier(n_estimators=500, class_weight="balanced", n_jobs=-1, random_state=i)
        rf.fit(X_boot, y_boot)
        importance_matrix_final[i] = rf.feature_importances_

    mean_imp_final = importance_matrix_final.mean(axis=0)
    ranking_final = pd.DataFrame({"protein": X_train.columns, "importance": mean_imp_final}).sort_values("importance", ascending=False)
    ranking_final["importance"] /= ranking_final["importance"].sum()
    ranking_final["cum_importance"] = ranking_final["importance"].cumsum()

    top_features = ranking_final[ranking_final["cum_importance"] <= best_threshold]["protein"]
    
    # TRACK EVERY FEATURE SELECTED IN THIS SEED
    for f in top_features:
        feature_counter[f] = feature_counter.get(f, 0) + 1

    # External Test (Lisbon)
    rf_final = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42)
    rf_final.fit(X_train[top_features], y_train)
    y_pred_lisbon = rf_final.predict(X_test[top_features])
    ad_idx_label = np.where(rf_final.classes_ == "MCI-AD")[0][0]
    y_prob_lisbon = rf_final.predict_proba(X_test[top_features])[:, ad_idx_label]

    seed_mcc = matthews_corrcoef(y_test, y_pred_lisbon)
    seed_auc = roc_auc_score(y_test_bin, y_prob_lisbon)

    # PER-SEED REPORT
    print(f"\n>> SEED {seed} RESULTS:")
    print(f"   Best Threshold: {best_threshold}")
    print(f"   Features Selected: {len(top_features)}")
    print(f"   Test MCC: {seed_mcc:.3f}")
    print(f"   Test AUC: {seed_auc:.3f}")

    all_results.append({
        "seed": seed, "mcc": seed_mcc, "auc": seed_auc, 
        "n_features": len(top_features), "threshold": best_threshold
    })

# ======================
# FINAL AGGREGATED REPORT
# ======================
df_res = pd.DataFrame(all_results)

print("\n" + "="*30)
print("FINAL SUMMARY (OVER ALL SEEDS)")
print("="*30)
print(f"Global Mean MCC: {df_res['mcc'].mean():.3f} (±{df_res['mcc'].std():.3f})")
print(f"Global Mean AUC: {df_res['auc'].mean():.3f} (±{df_res['auc'].std():.3f})")
print(f"Avg Features Selected: {df_res['n_features'].mean():.1f}")

print("\n--- Feature Frequency (Top 20) ---")
# Normalized frequency (0.0 to 1.0)
feat_freq_LisbononCoimbra = (pd.Series(feature_counter).sort_values(ascending=False) / len(seeds))
print(feat_freq_LisbononCoimbra.head(20))

print("\n--- Threshold Frequency ---")
thresh_freq = (pd.Series(threshold_counter).sort_index() / len(seeds))
print(thresh_freq)

ok

>> SEED 0 RESULTS:
   Best Threshold: 0.7
   Features Selected: 50
   Test MCC: 0.763
   Test AUC: 0.968

>> SEED 1 RESULTS:
   Best Threshold: 0.7
   Features Selected: 50
   Test MCC: 0.747
   Test AUC: 0.967

>> SEED 2 RESULTS:
   Best Threshold: 0.7
   Features Selected: 51
   Test MCC: 0.803
   Test AUC: 0.947

>> SEED 3 RESULTS:
   Best Threshold: 0.7
   Features Selected: 51
   Test MCC: 0.803
   Test AUC: 0.957

>> SEED 4 RESULTS:
   Best Threshold: 0.7
   Features Selected: 49
   Test MCC: 0.803
   Test AUC: 0.971

>> SEED 5 RESULTS:
   Best Threshold: 0.7
   Features Selected: 50
   Test MCC: 0.747
   Test AUC: 0.973

>> SEED 6 RESULTS:
   Best Threshold: 0.7
   Features Selected: 47
   Test MCC: 0.747
   Test AUC: 0.958

>> SEED 7 RESULTS:
   Best Threshold: 0.7
   Features Selected: 50
   Test MCC: 0.794
   Test AUC: 0.976

>> SEED 8 RESULTS:
   Best Threshold: 0.7
   Features Selected: 49
   Test MCC: 0.832
   Test AUC: 0.973

>> SEED 9 RESULTS:
   Best Threshold: 0.7


In [13]:
print(len(feat_freq_LisbononCoimbra))
feat_freq_LisbononCoimbra.to_pickle('feat_freq_LisbononCoimbra_rf_threshold.pkl')

87
